# Capstone — A GAN that generates digits (MNIST)

_Capstones_

**Walk five papers, building piece by piece a conditional DCGAN that draws whatever MNIST digit you ask for.**

---

This notebook is a practice scaffold for the **Capstone — A GAN that generates digits (MNIST)** lesson. We assemble the conditional DCGAN one piece at a time — the generator, the discriminator, the data pipeline, and the adversarial training loop — then watch a requested digit emerge from noise.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders/dropdowns, and outside Colab they fall back to a sensible default. Run each cell top to bottom, experiment with those knobs, and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.max_open_warning"] = 0

## First, look at the data

A conditional GAN learns a separate-looking image distribution for each label `0..9`, so we inspect the labels, image shapes, and pixel scaling before building a model. The generator will end in `Tanh`, which naturally outputs values in `[-1, 1]`; the real MNIST images must be normalized the same way.

### Step 1 — Load MNIST and inspect real samples

MNIST images are grayscale handwritten digits. The raw dataset stores PIL images and integer labels; the table shows a few labels and image sizes, and the grid lets us see the visual variety the generator must imitate.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import torchvision
import torchvision.transforms as T

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "digit labels 0-9"))

rows = []
for i in range(10):
    image, label = preview[i]
    rows.append({"index": i, "label": label, "width": image.size[0], "height": image.size[1], "mode": image.mode})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(2, 5, figsize=(8, 3.2))
for ax, i in zip(axes.ravel(), range(10)):
    image, label = preview[i]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"label {label}")
    ax.axis("off")
plt.suptitle("MNIST examples")
plt.tight_layout()
plt.show()

### Step 2 — Label distribution

A conditional model only learns a class well if it sees enough examples of that class. MNIST is close to balanced, so every requested digit gets plenty of training signal.

In [ ]:
labels = np.array(preview.targets)
counts = np.bincount(labels, minlength=10)
label_df = pd.DataFrame({"digit": np.arange(10), "count": counts, "percent": counts / counts.sum() * 100})
display(label_df)

plt.figure(figsize=(7, 3))
plt.bar(label_df["digit"], label_df["count"], color="#79c0ff")
plt.xticks(range(10)); plt.xlabel("digit label"); plt.ylabel("count")
plt.title("MNIST label counts")
plt.show()

### Step 3 — Pixel tensors and `[-1, 1]` normalization

`ToTensor()` converts a PIL image to a `(C,H,W)` tensor in `[0,1]`. We resize from `28x28` to `32x32` for the DCGAN stride pattern, then normalize with `(x - 0.5)/0.5` so real images live in `[-1,1]`, matching the generator's `Tanh` output.

In [ ]:
resize = T.Resize(32)
to_tensor = T.ToTensor()
normalize = T.Normalize((0.5,), (0.5,))
raw_img, raw_label = preview[0]
raw_32 = resize(raw_img)
raw_tensor = to_tensor(raw_32)
norm_tensor = normalize(raw_tensor)

shape_df = pd.DataFrame([
    {"stage": "PIL image", "shape": raw_img.size, "min": None, "max": None},
    {"stage": "ToTensor resized", "shape": tuple(raw_tensor.shape), "min": float(raw_tensor.min()), "max": float(raw_tensor.max())},
    {"stage": "Normalize", "shape": tuple(norm_tensor.shape), "min": float(norm_tensor.min()), "max": float(norm_tensor.max())},
])
display(shape_df)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(raw_tensor[0], cmap="gray", vmin=0, vmax=1); axes[0].set_title("image tensor")
axes[0].axis("off")
axes[1].hist(raw_tensor.flatten().numpy(), bins=30, color="#79c0ff"); axes[1].set_title("pixels [0,1]")
axes[2].hist(norm_tensor.flatten().numpy(), bins=30, color="#ffb86b"); axes[2].set_title("pixels [-1,1]")
for ax in axes[1:]:
    ax.set_xlabel("value"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()

## Reference implementation — PyTorch

The finished model is a **conditional DCGAN** trained with the original GAN's adversarial objective. Each ingredient comes from a paper:

| Paper | Contributes | In our notebook |
|---|---|---|
| GAN | generator vs discriminator min-max game | BCE losses for `D` and non-saturating `G` |
| DCGAN | convolutional generator/discriminator recipe | strided convs, transposed convs, Adam betas |
| BatchNorm | stable feature scaling in deep conv nets | `nn.BatchNorm2d` inside both networks |
| WGAN | intuition: smoother distances help GANs | we contrast BCE saturation with a critic-style score |
| cGAN | label-conditioned generation | digit embeddings threaded through `G(z,y)` and `D(x,y)` |

The hero operation is built tensor by tensor: a noise vector plus a label embedding becomes an image, and the discriminator scores real/fake image-label pairs for the adversarial losses.

### Step 4 — Set hyperparameters and re-check the GAN math

We keep the original seed and dimensions: `NZ=100`, `NGF=NDF=64`, `EMB=50`, one grayscale channel, and ten labels. At the GAN optimum the discriminator can only guess, so its two BCE terms sum to `2*log(2)`. The transposed-conv formula shows why kernel 4, stride 2, padding 1 doubles spatial size.

In [ ]:
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
K  = 10                                   # MNIST has 10 digit classes (0..9)
NZ, NGF, NDF, NC = 100, 64, 64, 1         # noise dim, generator/discriminator widths, image channels
EMB = 50                                  # size of the learned label embedding

# At the GAN optimum p_g = p_data the discriminator is reduced to guessing, D = 1/2.
def convT_out(h, k, s, p):
    return (h - 1) * s - 2 * p + k        # H_out = (H_in-1)s - 2p + k

equilibrium_loss = 2 * math.log(2)
print("equilibrium D-loss 2*log2 = %.4f   (= -log4 magnitude %.4f)" % (equilibrium_loss, math.log(4)))
doubled = convT_out(4, 4, 2, 1)
print("convT(4x4, k4 s2 p1) -> %dx%d (doubles)" % (doubled, doubled))
probe = nn.ConvTranspose2d(8, 8, 4, 2, 1)
assert convT_out(4, 4, 2, 1) == 8 == probe(torch.zeros(1, 8, 4, 4)).shape[-1]

### Step 5 — Hero op part A: noise plus label embedding

The generator's input is not just random noise. In a cGAN, the requested digit label gets its own learned embedding vector; we concatenate that embedding beside `z` as extra channels. The result is a `(batch, NZ+EMB, 1, 1)` tensor that says both "draw something random" and "make it this class."

In [ ]:
torch.manual_seed(0)
hero_y = torch.tensor([7])
hero_z = torch.randn(1, NZ, 1, 1)
hero_label_emb = nn.Embedding(K, EMB)
hero_e = hero_label_emb(hero_y).view(1, EMB, 1, 1)
hero_in = torch.cat([hero_z, hero_e], dim=1)

display(pd.DataFrame([
    {"tensor": "z noise", "shape": tuple(hero_z.shape), "mean": float(hero_z.mean()), "std": float(hero_z.std())},
    {"tensor": "label embedding", "shape": tuple(hero_e.shape), "mean": float(hero_e.detach().mean()), "std": float(hero_e.detach().std())},
    {"tensor": "concat input", "shape": tuple(hero_in.shape), "mean": float(hero_in.detach().mean()), "std": float(hero_in.detach().std())},
]))

plt.figure(figsize=(8, 2.5))
plt.plot(hero_z.flatten()[:40].numpy(), label="first 40 z dims")
plt.plot(np.arange(40, 50), hero_e.flatten()[:10].detach().numpy(), "o", label="first 10 label dims")
plt.title("noise and label features")
plt.legend(); plt.show()

### Step 6 — Hero op part B: transposed convs grow `1x1` into `32x32`

DCGAN replaces fully-connected image synthesis with a stack of transposed convolutions. We pass the concrete concatenated tensor through the exact upsampling recipe and record every shape: `1x1 -> 4x4 -> 8x8 -> 16x16 -> 32x32`. The final `Tanh` keeps pixels in `[-1,1]`.

In [ ]:
hero_G_layers = nn.Sequential(
    nn.ConvTranspose2d(NZ + EMB, NGF*4, 4, 1, 0, bias=False),
    nn.BatchNorm2d(NGF*4), nn.ReLU(True),
    nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
    nn.BatchNorm2d(NGF*2), nn.ReLU(True),
    nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),
    nn.BatchNorm2d(NGF),   nn.ReLU(True),
    nn.ConvTranspose2d(NGF, NC,      4, 2, 1, bias=False),
    nn.Tanh())
hero_G_layers.eval()

h = hero_in
shape_rows = [{"stage": "concat input", "shape": tuple(h.shape), "min": float(h.detach().min()), "max": float(h.detach().max())}]
with torch.no_grad():
    for i, layer in enumerate(hero_G_layers):
        h = layer(h)
        if isinstance(layer, (nn.ConvTranspose2d, nn.Tanh)):
            shape_rows.append({"stage": f"{i}: {layer.__class__.__name__}", "shape": tuple(h.shape),
                               "min": float(h.detach().min()), "max": float(h.detach().max())})
hero_fake = h

display(pd.DataFrame(shape_rows))
plt.figure(figsize=(3, 3))
plt.imshow(((hero_fake[0, 0].detach() + 1) / 2), cmap="gray", vmin=0, vmax=1)
plt.title("untrained G image")
plt.axis("off"); plt.show()

### Step 7 — BatchNorm in one picture

BatchNorm learns a scale and shift, but its immediate job is simple: normalize each feature channel so activations are easier to optimize. The table compares the first transposed-conv block before and after BatchNorm.

In [ ]:
conv0 = hero_G_layers[0]
bn0 = hero_G_layers[1]
with torch.no_grad():
    pre_bn = conv0(hero_in)
    post_bn = bn0(pre_bn)

stats = pd.DataFrame([
    {"tensor": "before BatchNorm", "mean": float(pre_bn.mean()), "std": float(pre_bn.std()), "min": float(pre_bn.min()), "max": float(pre_bn.max())},
    {"tensor": "after BatchNorm", "mean": float(post_bn.mean()), "std": float(post_bn.std()), "min": float(post_bn.min()), "max": float(post_bn.max())},
])
display(stats)

plt.figure(figsize=(7, 3))
plt.hist(pre_bn.flatten().detach().numpy(), bins=40, alpha=0.6, label="before")
plt.hist(post_bn.flatten().detach().numpy(), bins=40, alpha=0.6, label="after")
plt.title("BatchNorm activation scale")
plt.legend(); plt.show()

### Step 8 — Hero op part C: discriminator sees image plus label plane

The discriminator also receives the label. It embeds `y` into a full `32x32` plane and concatenates that plane as a second image channel. Then strided convolutions shrink `32x32 -> 16x16 -> 8x8 -> 4x4 -> 1x1`, producing one real/fake logit.

In [ ]:
tfm = T.Compose([T.Resize(32), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
# final notebook: use the full MNIST training set
real_x, real_y_value = data[0]
real_x = real_x.unsqueeze(0)
real_y = torch.tensor([real_y_value])

hero_D_label_emb = nn.Embedding(K, 32*32)
hero_D_layers = nn.Sequential(
    nn.Conv2d(NC + 1, NDF, 4, 2, 1, bias=False),
    nn.LeakyReLU(0.2, True),
    nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
    nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, True),
    nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
    nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, True),
    nn.Conv2d(NDF*4, 1, 4, 1, 0, bias=False))
hero_D_layers.eval()

label_plane = hero_D_label_emb(real_y).view(1, 1, 32, 32)
d_in = torch.cat([real_x, label_plane], dim=1)
h = d_in
rows = [{"stage": "image + label plane", "shape": tuple(h.shape)}]
with torch.no_grad():
    for i, layer in enumerate(hero_D_layers):
        h = layer(h)
        if isinstance(layer, nn.Conv2d):
            rows.append({"stage": f"{i}: Conv2d", "shape": tuple(h.shape)})
real_logit_untrained = h.view(-1)

display(pd.DataFrame(rows))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow((real_x[0, 0] + 1) / 2, cmap="gray", vmin=0, vmax=1); axes[0].set_title(f"real y={real_y_value}")
axes[1].imshow(label_plane[0, 0].detach(), cmap="magma"); axes[1].set_title("label plane")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("untrained real logit:", round(float(real_logit_untrained.item()), 3), "sigmoid:", round(float(torch.sigmoid(real_logit_untrained).item()), 3))

### Step 9 — Package the conditional generator and discriminator

Now we wrap the exact operations into reusable modules. `G(z,y)` concatenates a `1x1` label embedding to noise. `D(x,y)` concatenates a `32x32` label plane to the image. `init_weights` applies the DCGAN normal initialization.

In [ ]:
# --- CONDITIONAL DCGAN GENERATOR: [noise | label-plane] -> 32x32 image. ---
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(K, EMB)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ + EMB, NGF*4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*4), nn.ReLU(True),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2), nn.ReLU(True),
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),   nn.ReLU(True),
            nn.ConvTranspose2d(NGF, NC,      4, 2, 1, bias=False),
            nn.Tanh())
    def forward(self, z, y):
        e = self.label_emb(y).view(z.size(0), EMB, 1, 1)
        return self.net(torch.cat([z, e], dim=1))

# --- CONDITIONAL DCGAN DISCRIMINATOR: [image | label-plane] -> one real/fake logit. ---
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(K, 32*32)
        self.net = nn.Sequential(
            nn.Conv2d(NC + 1, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*4, 1, 4, 1, 0, bias=False))
    def forward(self, x, y):
        e = self.label_emb(y).view(x.size(0), 1, 32, 32)
        return self.net(torch.cat([x, e], dim=1)).view(-1)

# DCGAN weight init: zero-centered normal, std 0.02.
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

### Step 10 — Instantiate, count parameters, and check random outputs

Before training, the generator should output noise-like images and the discriminator should score real/fake pairs near chance. The parameter table shows where capacity lives: most weights are in convolutional blocks, plus a small label-embedding budget.

In [ ]:
torch.manual_seed(0)
G = Generator().to(device); G.apply(init_weights)
D = Discriminator().to(device); D.apply(init_weights)

def grouped_params(model):
    groups = {}
    for name, p in model.named_parameters():
        group = "label_emb" if name.startswith("label_emb") else name.split(".")[0]
        groups[group] = groups.get(group, 0) + p.numel()
    return groups

rows = []
for model_name, model in [("G", G), ("D", D)]:
    groups = grouped_params(model)
    total = sum(groups.values())
    for group, n in groups.items():
        rows.append({"model": model_name, "component": group, "parameters": n, "% of model": round(100*n/total, 1)})
display(pd.DataFrame(rows))
print("G params:", f"{sum(p.numel() for p in G.parameters()):,}")
print("D params:", f"{sum(p.numel() for p in D.parameters()):,}")

with torch.no_grad():
    z0 = torch.randn(8, NZ, 1, 1, device=device)
    y0 = torch.arange(8, device=device) % K
    fake0 = G(z0, y0)
    logits0 = D(fake0, y0)
print("untrained fake batch:", tuple(fake0.shape), "pixel range", (round(float(fake0.min()), 3), round(float(fake0.max()), 3)))
print("untrained D sigmoid mean on fakes:", round(float(torch.sigmoid(logits0).mean()), 3))

### Step 11 — Build the dataloader and fixed visualization noise

MNIST is resized and normalized exactly as in the data section. We use BCE-with-logits, one Adam optimizer per network, DCGAN betas `(0.5, 0.999)`, and fixed noise so the same latent input can be visualized across training.

In [ ]:
loader = torch.utils.data.DataLoader(data, batch_size=128, shuffle=True, drop_last=True)
bce = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_z = torch.randn(1, NZ, 1, 1, device=device)             # same noise -> compare across epochs
fixed_z_by_label = fixed_z.repeat(K, 1, 1, 1)                 # same z, labels 0..9 -> condition test
fixed_labels = torch.arange(K, device=device)

def show_image_tensor(img, title="generated"):
    img = ((img.detach().cpu() + 1) / 2).clamp(0, 1)
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(img[0], cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off"); plt.show()

def preview(want):
    G.eval()
    with torch.no_grad():
        img = G(fixed_z, torch.tensor([want], device=device))[0].cpu()
    G.train()
    show_image_tensor(img, title=f"requested {want}")

### Step 12 — One adversarial training step

A single step has two updates. `D` learns **real with true label -> 1** and **fake with generated label -> 0**. Then `G` uses the non-saturating objective: make `D(fake, label)` predict **1**. This is the core GAN game in one table and one bar chart.

In [ ]:
x_step, y_step = next(iter(loader))
x_step, y_step = x_step.to(device), y_step.to(device)
n = x_step.size(0)
real_t = torch.ones(n, device=device)
fake_t = torch.zeros(n, device=device)

z = torch.randn(n, NZ, 1, 1, device=device)
yf = torch.randint(0, K, (n,), device=device)
fake = G(z, yf)
real_logits = D(x_step, y_step)
fake_logits = D(fake.detach(), yf)
lossD_real = bce(real_logits, real_t)
lossD_fake = bce(fake_logits, fake_t)
lossD = lossD_real + lossD_fake
optD.zero_grad(); lossD.backward(); optD.step()

lossG = bce(D(fake, yf), real_t)
optG.zero_grad(); lossG.backward(); optG.step()

step_df = pd.DataFrame([
    {"term": "D real -> 1", "loss": float(lossD_real.detach()), "mean sigmoid": float(torch.sigmoid(real_logits).mean().detach())},
    {"term": "D fake -> 0", "loss": float(lossD_fake.detach()), "mean sigmoid": float(torch.sigmoid(fake_logits).mean().detach())},
    {"term": "G fake -> 1", "loss": float(lossG.detach()), "mean sigmoid": float(torch.sigmoid(D(fake.detach(), yf)).mean().detach())},
])
display(step_df)
plt.figure(figsize=(5.5, 3))
plt.bar(step_df["term"], step_df["loss"], color=["#79c0ff", "#ff7b72", "#7ee787"])
plt.ylabel("BCE loss"); plt.title("one GAN step losses")
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

### Step 13 — WGAN intuition: BCE can saturate, distances need not

This notebook trains the classic DCGAN/cGAN BCE objective, but WGAN is part of the GAN story because it reframed training around a smoother critic distance. The plot below shows why: when logits get very confident, BCE gradients can become tiny for the saturating generator objective, while the non-saturating trick keeps a useful signal.

In [ ]:
logits = torch.linspace(-8, 8, 200)
prob = torch.sigmoid(logits)
saturating_G = -torch.log((1 - prob).clamp_min(1e-8))
non_sat_G = -torch.log(prob.clamp_min(1e-8))
critic_like = -logits

plt.figure(figsize=(7, 3.5))
plt.plot(logits.numpy(), saturating_G.numpy(), label="saturating G")
plt.plot(logits.numpy(), non_sat_G.numpy(), label="non-sat G")
plt.plot(logits.numpy(), (critic_like - critic_like.min()).numpy() / 2, label="critic-style", ls="--")
plt.xlabel("D logit on fake"); plt.ylabel("relative loss")
plt.title("GAN loss shapes")
plt.legend(); plt.show()

### Step 14 — The full adversarial loop (recording history)

For the real run we reinitialize from the original seed so the curve starts cleanly. Labels are threaded through both networks on every batch. We record `D` loss, `G` loss, and occasional fixed-noise snapshots. This is **our run**; exact images and losses vary by hardware.

In [ ]:
def make_models():
    torch.manual_seed(0)
    g = Generator().to(device); g.apply(init_weights)
    d = Discriminator().to(device); d.apply(init_weights)
    return g, d

G, D = make_models()
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_z = torch.randn(1, NZ, 1, 1, device=device)
fixed_z_by_label = fixed_z.repeat(K, 1, 1, 1)
fixed_labels = torch.arange(K, device=device)
loss_hist, snapshot_log = [], []

def generate_grid(labels, z=None):
    G.eval()
    labels = torch.as_tensor(labels, device=device, dtype=torch.long)
    if z is None:
        z = torch.randn(len(labels), NZ, 1, 1, device=device)
    with torch.no_grad():
        imgs = G(z, labels).detach().cpu()
    G.train()
    return imgs

def train(epochs=3):
    real_mean = next(iter(loader))[0].mean().item()
    print("real data pixel mean ~ %.3f  (target for generated samples)\n" % real_mean)
    step = 0
    for ep in range(epochs):
        for batch_i, (x, y) in zip(range(1000), loader):
            x = x.to(device)
            y = y.to(device)
            n = x.size(0)
            real_t = torch.ones(n, device=device)
            fake_t = torch.zeros(n, device=device)
            # (a) DISCRIMINATOR step: reals under true labels -> 1; fakes under generated labels -> 0.
            z  = torch.randn(n, NZ, 1, 1, device=device)
            yf = torch.randint(0, K, (n,), device=device)
            fake = G(z, yf)
            lossD = bce(D(x, y), real_t) + bce(D(fake.detach(), yf), fake_t)
            optD.zero_grad(); lossD.backward(); optD.step()
            # (b) GENERATOR step: NON-SATURATING -- make D call class-yf fakes "real for yf".
            lossG = bce(D(fake, yf), real_t)
            optG.zero_grad(); lossG.backward(); optG.step()
            loss_hist.append({"step": step, "D loss": float(lossD.item()), "G loss": float(lossG.item())})
            if step % 200 == 0:
                with torch.no_grad():
                    s = G(fixed_z, torch.tensor([ep % K], device=device))
                snapshot_log.append({"step": step, "digit": ep % K, "mean": float(s.mean()), "std": float(s.std())})
                print("step %5d  D %.3f  G %.3f  sample(mean %+.3f std %.3f)"
                      % (step, lossD.item(), lossG.item(), s.mean().item(), s.std().item()))
            step += 1
        preview(want=ep % K)

print("before training (requesting a 7) -- pure noise:")
preview(7)
train(epochs=3)
print("Now generate any digit on demand (our run):")
for d in [0, 5, 9]:
    preview(d)
print("D loss often hovers around 2*log2 = 1.386 when the game is balanced; this is our small run.")

## Visualize the data & results

_Did the conditional DCGAN learn to draw a requested MNIST digit, and did the adversarial game stay trackable?_ We answer using the history and trained generator already produced — no retraining inside the plots or sandboxes.

### Step 15 — D-loss and G-loss curves

GAN losses are a game, not a single supervised error, so we plot both curves. `D` and `G` can trade off: if `D` gets too strong, `G` loss rises; if `G` fools `D`, `D` loss rises.

In [ ]:
hist_df = pd.DataFrame(loss_hist)
display(hist_df.head())

plt.figure(figsize=(7.5, 3.5))
plt.plot(hist_df["step"], hist_df["D loss"], label="D loss", color="#ff7b72")
plt.plot(hist_df["step"], hist_df["G loss"], label="G loss", color="#7ee787")
plt.axhline(2 * math.log(2), ls="--", color="#999999", label="2 log 2")
plt.xlabel("training step"); plt.ylabel("BCE loss")
plt.title("GAN training losses")
plt.legend(); plt.show()

### Step 16 — Conditional grid: ask for digits 0 through 9

Here we reuse the same noise vector for every label. If conditioning is working, changing only `y` should change the requested digit identity.

In [ ]:
with torch.no_grad():
    label_grid = G(fixed_z_by_label, fixed_labels).detach().cpu()

fig, axes = plt.subplots(1, 10, figsize=(12, 1.7))
for d, ax in enumerate(axes):
    ax.imshow(((label_grid[d, 0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(str(d)); ax.axis("off")
plt.suptitle("same z, labels 0-9")
plt.tight_layout(); plt.show()

### Step 17 — Per-label sample grid

Now each label gets multiple fresh noise vectors. Rows are labels and columns are different `z` samples, so this grid checks both controllability (row identity) and diversity (columns vary).

In [ ]:
G.eval()
cols = 5
labels_many = torch.arange(K, device=device).repeat_interleave(cols)
z_many = torch.randn(K * cols, NZ, 1, 1, device=device)
with torch.no_grad():
    imgs_many = G(z_many, labels_many).detach().cpu()
G.train()

fig, axes = plt.subplots(K, cols, figsize=(cols * 1.4, K * 1.15))
for r in range(K):
    for c in range(cols):
        idx = r * cols + c
        axes[r, c].imshow(((imgs_many[idx, 0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_ylabel(str(r), rotation=0, labelpad=10)
plt.suptitle("per-label samples")
plt.tight_layout(); plt.show()

### Step 18 — Discriminator scores on real and generated images

A trained discriminator returns logits; `sigmoid(logit)` is the model's current "realness" score. We compare a small batch of real MNIST examples with generated examples under their labels.

In [ ]:
G.eval(); D.eval()
x_eval, y_eval = next(iter(loader))
x_eval, y_eval = x_eval[:32].to(device), y_eval[:32].to(device)
z_eval = torch.randn(32, NZ, 1, 1, device=device)
y_fake_eval = torch.arange(32, device=device) % K
with torch.no_grad():
    fake_eval = G(z_eval, y_fake_eval)
    real_scores = torch.sigmoid(D(x_eval, y_eval)).cpu().numpy()
    fake_scores = torch.sigmoid(D(fake_eval, y_fake_eval)).cpu().numpy()
G.train(); D.train()

score_df = pd.DataFrame({"kind": ["real"] * len(real_scores) + ["generated"] * len(fake_scores),
                         "score": np.r_[real_scores, fake_scores]})
display(score_df.groupby("kind")["score"].agg(["mean", "std", "min", "max"]).reset_index())
plt.figure(figsize=(6, 3))
plt.hist(real_scores, bins=12, alpha=0.65, label="real")
plt.hist(fake_scores, bins=12, alpha=0.65, label="generated")
plt.xlabel("D realness score"); plt.ylabel("count")
plt.title("D scores")
plt.legend(); plt.show()

**🎛️ Play with it — pick a digit label**

Choose the label sent into the trained generator. **Try:** each digit `0..9`. **Watch:** the requested class changes while the sampling code stays the same. **Why it matters:** this is the cGAN promise — steer generation with `y`.

In [ ]:
def play(digit=7):
    G.eval()
    with torch.no_grad():
        img = G(fixed_z, torch.tensor([digit], device=device))[0, 0].detach().cpu()
    G.train()
    plt.figure(figsize=(2.6, 2.6))
    plt.imshow(((img + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    plt.title(f"requested {digit}")
    plt.axis("off"); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, digit=Dropdown(options=list(range(10)), value=7))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — noise-vector sliders**

Two latent dimensions are exposed as sliders while the digit label stays fixed. **Try:** move `z0` and `z1`. **Watch:** style details change without retraining. **Why it matters:** `z` controls variation within a class, while `y` controls the class.

In [ ]:
base_z = fixed_z.detach().clone()

def play(digit=3, z0=0.0, z1=0.0):
    z = base_z.clone()
    z[0, 0, 0, 0] = z0
    z[0, 1, 0, 0] = z1
    G.eval()
    with torch.no_grad():
        img = G(z, torch.tensor([digit], device=device))[0, 0].detach().cpu()
    G.train()
    plt.figure(figsize=(2.6, 2.6))
    plt.imshow(((img + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    plt.title(f"digit {digit}, z sliders")
    plt.axis("off"); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, digit=Dropdown(options=list(range(10)), value=3),
             z0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.25),
             z1=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.25))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — interpolate between two noise vectors**

A row of images blends linearly from `z_a` to `z_b` for one fixed label. **Try:** change the digit. **Watch:** the handwriting style morphs smoothly if the latent space is organized. **Why it matters:** good generators turn nearby latent vectors into related images.

In [ ]:
torch.manual_seed(123)
z_a = torch.randn(1, NZ, 1, 1, device=device)
z_b = torch.randn(1, NZ, 1, 1, device=device)

def play(digit=8):
    alphas = torch.linspace(0, 1, 8, device=device).view(-1, 1, 1, 1)
    z = (1 - alphas) * z_a + alphas * z_b
    y = torch.full((8,), digit, device=device, dtype=torch.long)
    G.eval()
    with torch.no_grad():
        imgs = G(z, y).detach().cpu()
    G.train()
    fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
    for i, ax in enumerate(axes):
        ax.imshow(((imgs[i, 0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{i}/7"); ax.axis("off")
    plt.suptitle(f"interpolate digit {digit}")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, digit=Dropdown(options=list(range(10)), value=8))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — discriminator score real vs generated**

Pick a digit and compare one cached real MNIST image with one generated image under that label. **Try:** labels where the generator looks good or bad. **Watch:** whether `D` agrees. **Why it matters:** `D` supplies the training signal that pushes `G` toward realistic class-conditional images.

In [ ]:
first_by_digit = {}
for idx, (_, label) in enumerate(data):
    if label not in first_by_digit:
        first_by_digit[label] = idx
    if len(first_by_digit) == 10:
        break

def play(digit=4):
    real_img, _ = data[first_by_digit[digit]]
    real_img = real_img.unsqueeze(0).to(device)
    y = torch.tensor([digit], device=device)
    z = base_z.clone()
    G.eval(); D.eval()
    with torch.no_grad():
        fake_img = G(z, y)
        real_score = torch.sigmoid(D(real_img, y)).item()
        fake_score = torch.sigmoid(D(fake_img, y)).item()
    G.train(); D.train()
    fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
    axes[0].imshow(((real_img[0, 0].cpu() + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"real {real_score:.2f}")
    axes[1].imshow(((fake_img[0, 0].detach().cpu() + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"gen {fake_score:.2f}")
    for ax in axes: ax.axis("off")
    plt.suptitle(f"D scores for label {digit}")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, digit=Dropdown(options=list(range(10)), value=4))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — same noise across all labels**

Hold `z` fixed and generate a row for labels `0..9`. **Try:** different latent seed choices. **Watch:** the row changes style, but each column still asks for a different digit. **Why it matters:** it separates the roles of `z` (style) and `y` (class).

In [ ]:
def play(seed=0):
    gen = torch.Generator(device=device).manual_seed(int(seed))
    z = torch.randn(1, NZ, 1, 1, device=device, generator=gen).repeat(K, 1, 1, 1)
    labels_row = torch.arange(K, device=device)
    G.eval()
    with torch.no_grad():
        imgs = G(z, labels_row).detach().cpu()
    G.train()
    fig, axes = plt.subplots(1, 10, figsize=(12, 1.7))
    for d, ax in enumerate(axes):
        ax.imshow(((imgs[d, 0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(str(d)); ax.axis("off")
    plt.suptitle(f"same z seed {seed}")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, seed=IntSlider(value=0, min=0, max=20, step=1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — G vs D loss tradeoff**

Use the recorded history instead of retraining. **Try:** choose a smoothing window. **Watch:** short-term spikes and longer trends in the two-player game. **Why it matters:** GAN progress is diagnosed by both losses together, not just one scalar going down.

In [ ]:
def play(window=5):
    df = hist_df.copy()
    w = max(1, int(window))
    df["D smooth"] = df["D loss"].rolling(w, min_periods=1).mean()
    df["G smooth"] = df["G loss"].rolling(w, min_periods=1).mean()
    plt.figure(figsize=(7.5, 3.5))
    plt.plot(df["step"], df["D smooth"], label="D", color="#ff7b72")
    plt.plot(df["step"], df["G smooth"], label="G", color="#7ee787")
    plt.axhline(2 * math.log(2), ls="--", color="#999999", label="2 log 2")
    plt.xlabel("training step"); plt.ylabel("smoothed loss")
    plt.title(f"loss tradeoff window {w}")
    plt.legend(); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, window=IntSlider(value=5, min=1, max=50, step=1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Practice

1. Replace the BCE objective with a WGAN-style critic loss. What code changes are needed in the discriminator output, optimizer step, and weight clipping or gradient penalty?
2. Remove the label embeddings from both networks to make a plain DCGAN. What generation knob disappears?
3. Increase or decrease `NZ`. Does the generator use the extra latent dimensions in a visible way?
4. Save fixed-noise grids at the end of every epoch and make a training-progress animation.